1. найти изображение и создать для нее маску
2. аугментация (случайно повышать резкость, поворачивать, обрезка и тп) сделать n раз (пусть 100)

In [64]:
import cv2
import numpy as np
import os
import random
import shutil  # для удаления папки с содержимым


from PIL import Image

In [45]:



image = cv2.imread('2.jpg')
h, w = image.shape[:2]

mask = np.zeros((h, w), dtype=np.uint8)

x1 = 550
y1 = 20
x2 = 900
y2 = 330

cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)

masked_image = cv2.bitwise_and(image, image, mask=mask)

cv2.imwrite('masked_img.jpg', masked_image)



True

In [46]:
def show_image(img, max_size=312):
    h, w = img.shape[:2]
    scale = min(max_size / w, max_size / h)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (new_w, new_h))
    if len(resized.shape) == 3:
        img_rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
        display(Image.fromarray(img_rgb))
    else:
        display(Image.fromarray(resized))

In [47]:
masked_img = cv2.imread('masked_img.jpg')
orig_img = cv2.imread('2.jpg')

In [71]:
def aug(orig_img, masked_img, n=5):
    if os.path.exists('augmented'):
        shutil.rmtree('augmented')

    os.makedirs("augmented/images", exist_ok=True)
    os.makedirs("augmented/masks", exist_ok=True)

    for i in range(n):
        img = orig_img.copy()
        mask = masked_img.copy()

        random_size = random.randint(200, 400)
        random_angle = random.randint(0, 360)
        random_brightness = random.uniform(-50, 50)
        random_centerTransformX = random.randint(0, 50)
        random_centerTransformY = random.randint(0, 50)

        # 1 ресайз
        img = cv2.resize(img, (random_size, random_size))
        mask = cv2.resize(mask, (random_size, random_size))

        # 2 (сделана) (сделана) поворот
        h, w = img.shape[:2]
        center = (w//2, h//2)
        matrix = cv2.getRotationMatrix2D(center, random_angle, 1.0)
        img = cv2.warpAffine(img, matrix, (w, h))
        mask = cv2.warpAffine(mask, matrix, (w, h))


        # 3 (сделана) яркость
        img = cv2.convertScaleAbs(img, alpha=1.0, beta=random_brightness)


        # 4 переместить центр изображения
        tx, ty = random_centerTransformX, random_centerTransformY

        # матрица трансляции
        M = np.float32([[1, 0, tx],
                        [0, 1, ty]])
        img = cv2.warpAffine(img, M, (w, h))
        mask = cv2.warpAffine(mask, M, (w, h) )

        name = f"{i:04d}_size{random_size}_angle{random_angle}_bright{int(random_brightness)}_shifted{random_centerTransformX},{random_centerTransformY}"
        cv2.imwrite(f"augmented/images/{name}.jpg", img)
        cv2.imwrite(f"augmented/masks/{name}.jpg", mask)

In [72]:
aug(orig_img=orig_img, masked_img=masked_img)